# Qwen3.5-0.8B serving smoke experiment

This notebook is the analysis and launch surface for `qwen35-0.8b-smoke`. It loads the real YAML profile, expands the same matrix as the CLI, reads the harness's canonical `manifest.json` and `summary.json` artifacts, and keeps quality evidence separate from performance.

**Safe default:** Run All is offline and read-only. It does not connect over SSH, provision a Pod, invoke the runner, or make a paid call. The launch cell runs only after `RUN_EXPERIMENT = True` is set deliberately.

In [1]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import Markdown, display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'experiment_profiles' / 'qwen35-0.8b-smoke.yaml').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from within the llm-serving checkout.')

PROJECT_ROOT = find_project_root(Path.cwd())
PROFILE_PATH = PROJECT_ROOT / 'experiment_profiles' / 'qwen35-0.8b-smoke.yaml'
OUTPUT_ROOT = Path(os.environ.get('LLM_SERVING_OUTPUT_ROOT', PROJECT_ROOT / 'output')).expanduser().resolve()
FIXTURE_ROOT = PROJECT_ROOT / 'notebooks' / 'fixtures'

from llm_serving.analysis import (
    comparable_quality_groups, discover_runs, history_rows, performance_rows,
    plot_baseline_deltas, plot_performance, plot_quality, quality_rows, write_run_notes,
)
from llm_serving.matrix import expand_matrix
from llm_serving.schemas import ExperimentProfile

print(f'Project: {PROJECT_ROOT}')
print(f'Artifact root: {OUTPUT_ROOT}')

Project: /home/kevin/Workspace/toyai-worktrees/experiment-notebook/llm-serving
Artifact root: /home/kevin/Workspace/toyai-worktrees/experiment-notebook/llm-serving/output


## Question and hypothesis

The smoke profile asks whether CUDA graph execution improves Qwen3.5-0.8B serving performance for short-interactive, long-prefill, near-limit-prefill, and decode-heavy traffic while chunked prefill remains enabled for the model's aligned Mamba cache mode.

The working hypothesis is that CUDA graphs increase output throughput and reduce inter-token latency most clearly on decode-heavy traffic. GSM8K is a separate smoke-quality check; its 20-example score is not a full benchmark and must not be used to explain a performance delta.

In [2]:
profile_raw = yaml.safe_load(PROFILE_PATH.read_text(encoding='utf-8'))
profile = ExperimentProfile.model_validate(profile_raw)
display(Markdown(f'**Loaded profile:** `{PROFILE_PATH}`  \n**Model:** `{profile.model.id}` at revision `{profile.model.revision}`'))
pd.json_normalize(profile.model_dump(), sep='.').T.rename(columns={0: 'resolved value'})

**Loaded profile:** `/home/kevin/Workspace/toyai-worktrees/experiment-notebook/llm-serving/experiment_profiles/qwen35-0.8b-smoke.yaml`  
**Model:** `Qwen/Qwen3.5-0.8B` at revision `main`

,resolved value
schema_version,1
name,qwen35-0.8b-smoke
model.id,Qwen/Qwen3.5-0.8B
model.revision,main
model.variant,BF16
model.dtype,auto
model.quantization,None
parallelism.tensor_parallel,1
parallelism.data_parallel,1
server.max_model_len,16384


## Resolved experiment matrix

This uses the harness's matrix expansion rather than reimplementing it in the notebook. The first row is the within-run baseline used by the saved reports.

In [3]:
matrix = expand_matrix(profile)
matrix_df = pd.DataFrame([
    {
        'case_id': case.case_id, 'baseline': case.is_baseline,
        'cuda_graphs': case.cuda_graphs, 'chunked_prefill': case.chunked_prefill,
        'repetition': case.repetition, 'vllm_args': shlex.join(case.vllm_args),
    }
    for case in matrix
])
matrix_df

,case_id,baseline,cuda_graphs,chunked_prefill,repetition,vllm_args
0,case-00-cg1-cp1-rep0,True,True,True,0,--model Qwen/Qwen3.5-0.8B --revision main --te...
1,case-01-cg0-cp1-rep0,False,False,True,0,--model Qwen/Qwen3.5-0.8B --revision main --te...


## Explicit launch controls

Edit the host alias and inventory path for an existing SSH host or for the temporary host inventory generated by the local Flash/Runpod driver. The command previews are inert. The existing harness command runs only when the switch in the next cell is changed to `True`.

Set the hypothesis before launch so the new run receives an `experiment-notes.json` sidecar. The notebook creates that sidecar only for the newly created run and refuses to replace an existing notes file.

In [ ]:
HOST = os.environ.get('LLM_SERVING_HOST', 'replace-with-host-alias')
HOST_INVENTORY = Path(os.environ.get('LLM_SERVING_HOST_INVENTORY', PROJECT_ROOT / 'hosts.local.yaml')).expanduser()
HYPOTHESIS = 'CUDA graphs improve decode-heavy throughput and ITL on this host.'
INITIAL_OBSERVATIONS = ''

plan_command = [sys.executable, '-m', 'llm_serving.cli', 'plan', str(PROFILE_PATH), '--host', HOST, '--inventory', str(HOST_INVENTORY)]
run_command = [sys.executable, '-m', 'llm_serving.cli', 'run', str(PROFILE_PATH), '--host', HOST, '--inventory', str(HOST_INVENTORY), '--output-dir', str(OUTPUT_ROOT)]
FLASH_CONFIG = PROJECT_ROOT / 'runpod.local.yaml'
flash_plan_example = ['uv', 'run', 'flash', '--config', str(FLASH_CONFIG), 'plan']
flash_create_example = ['uv', 'run', 'flash', '--config', str(FLASH_CONFIG), 'create', '--wait']
flash_stop_example = ['uv', 'run', 'flash', '--config', str(FLASH_CONFIG), 'stop', 'POD_ID']
display(Markdown('**Offline plan preview**\n```shell\n' + shlex.join(plan_command) + '\n```'))
display(Markdown('**Existing-harness run preview**\n```shell\n' + shlex.join(run_command) + '\n```'))
display(Markdown('**Flash lifecycle previews (do not run here)**\n```shell\n' + '\n'.join(map(shlex.join, [flash_plan_example, flash_create_example, flash_stop_example])) + '\n```\n`create --wait` emits `.flash/hosts/<pod-id>.yaml`; use it as `LLM_SERVING_HOST_INVENTORY` with host alias `runpod-qwen08`. Stop the Pod explicitly after retrieving artifacts.'))

In [ ]:
RUN_EXPERIMENT = False  # Deliberate opt-in: may use SSH/GPU resources and incur cost.

if not RUN_EXPERIMENT:
    print('Analysis-only mode: no validation, SSH, provisioning, or experiment run was started.')
else:
    if HOST == 'replace-with-host-alias':
        raise ValueError('Set LLM_SERVING_HOST or edit HOST before launching.')
    if not HOST_INVENTORY.is_file():
        raise FileNotFoundError(f'Host inventory not found: {HOST_INVENTORY}')
    before = {p.parent for p in OUTPUT_ROOT.rglob('summary.json')} if OUTPUT_ROOT.exists() else set()
    completed = subprocess.run(run_command, cwd=PROJECT_ROOT, check=False)
    after = {p.parent for p in OUTPUT_ROOT.rglob('summary.json')}
    created = sorted(after - before)
    if len(created) == 1:
        notes_path = write_run_notes(created[0], hypothesis=HYPOTHESIS, observations=INITIAL_OBSERVATIONS)
        print(f'Run notes created: {notes_path}')
    else:
        raise RuntimeError(f'Expected one new run directory, found {created}')
    print(f'Harness exit code: {completed.returncode}. Failed runs remain available for analysis.')

## Saved run history

The reader includes failed and partial runs. Missing files and metrics remain visible warnings or blank values; they are never replaced with zero. If the selected output root has no measured performance rows, the notebook uses bundled **synthetic fixtures** to exercise the tables and charts.

In [ ]:
measured_runs = discover_runs(OUTPUT_ROOT, profile_name=profile.name, source='measured')
fixture_runs = discover_runs(FIXTURE_ROOT, profile_name=profile.name, source='synthetic fixture')
history = measured_runs if measured_runs else fixture_runs
if measured_runs:
    display(Markdown(f'**Measured artifact history:** {len(measured_runs)} run(s) found.'))
else:
    display(Markdown('> **Synthetic fixture history shown.** No measured runs were found under the selected output root. Fixture values demonstrate the analysis only; they are not benchmark results.'))
pd.DataFrame(history_rows(history))

## Performance results

These panels show throughput, TTFT, ITL, and end-to-end latency only where the artifacts record them. Blue bars are baseline cases and orange bars are alternatives. Variability is not drawn because the current profile has one repetition per case; increase repetitions before interpreting run-to-run spread.

In [ ]:
measured_performance = performance_rows(measured_runs)
requested_ids = {value.strip() for value in os.environ.get('LLM_SERVING_RUN_IDS', '').split(',') if value.strip()}
if measured_performance:
    if requested_ids:
        analysis_runs = [run for run in measured_runs if run['run_id'] in requested_ids]
        if not performance_rows(analysis_runs):
            raise ValueError(f'LLM_SERVING_RUN_IDS selected no runs with performance rows: {sorted(requested_ids)}')
    else:
        analysis_runs = [next((run for run in measured_runs if run['complete'] and performance_rows([run])), next(run for run in measured_runs if performance_rows([run])))]
    display(Markdown('**Data source: measured harness artifacts.** Charts show the latest complete run by default; set `LLM_SERVING_RUN_IDS` to a comma-separated list for an explicit comparison.'))
else:
    analysis_runs = fixture_runs
    display(Markdown('> **Data source: synthetic fixtures.** The following values are illustrative and must not be reported as measurements.'))
perf = performance_rows(analysis_runs)
perf_df = pd.DataFrame(perf)
columns = ['run_id', 'source', 'case_id', 'case_status', 'config', 'workload', 'output_throughput_tok_per_s', 'ttft_p50_ms', 'itl_p50_ms', 'e2e_p50_ms']
perf_df.reindex(columns=columns)

In [ ]:
plot_performance(perf);


## Within-run baseline deltas

Deltas come directly from each canonical summary. Throughput gains are favorable; latency reductions are favorable. Cross-run deltas are not synthesized when hardware, runtime, or workload identity may differ.

In [ ]:
plot_baseline_deltas(perf);


## Quality evidence (separate from performance)

Quality rows are grouped only when their recorded task settings and named metric match. That check cannot establish compatibility for metadata the current artifacts do not record, such as evaluator version or prompt-template revision. The 20-example GSM8K setting is a smoke subset, not a leaderboard estimate.

In [ ]:
measured_quality = quality_rows(measured_runs)
if measured_quality:
    quality = measured_quality
    display(Markdown('**Data source: measured quality artifacts.**'))
else:
    quality = quality_rows(fixture_runs)
    display(Markdown('> **Data source: synthetic quality fixture.** No measured quality scores were found; this row demonstrates the schema only.'))
quality_df = pd.DataFrame(quality)
display(quality_df)
groups = comparable_quality_groups(quality)
print(f'Groups sharing recorded task settings and metric: {len(groups)}')
plot_quality(quality);

## Observations and next decision

After a real run, record what happened in that run's `experiment-notes.json` sidecar and rerun the analysis cells. Check completion and warning columns before comparing values. Treat missing E2E or quality values as unavailable. With one repetition, use the result to choose a focused repeat rather than claim stable variability.

Suggested decision: if the measured decode-heavy delta agrees with the hypothesis, repeat both cases at least three times on the same host/runtime. If a case failed, inspect its saved server log and keep the partial run in history.